# SpendDNA — Rahul Sharma's Spending Analysis
**Name:** Sandhya A

**Batch:** JUNE 2026

**Date:** 05-07-2026

Industry-graded minor project analyzing 6 months of synthetic UPI transaction data.

###AI Assistant like claude is used to code structure and in debugging the errors in code.



---




# #Feature 1 - The Transaction Parser

In [65]:
import pandas as pd
import numpy as np


In [66]:
df= pd.read_csv("/content/Data set for DADS June.csv")
print(df.shape)
print(df.dtypes)
df.head(10)

(1328, 8)
Date            object
Time            object
Description     object
Type            object
Amount          object
Balance        float64
Mode            object
Ref             object
dtype: object


,Date,Time,Description,Type,Amount,Balance,Mode,Ref
0,2024-01-01,03:11,AMAZON SELLER SVCS,Debit,₹2462,678275.0,UPI,TXN190872
1,01-Jan-24,05:44,BHIM-BMTC,DR,50.00,681007.0,UPI,TXN143064
2,01-Jan-24,09:35,NEFT-TECHCRUSH LABS-SALARY MAY24,CR,₹84728,484728.0,NEFT,TXN246316
3,2024-01-01,14:07,UPI-AMAN-8934@OKAXIS,Debit,₹1828,-748745.0,UPI,TXN569226
4,01 Jan 2024,14:23,BHIM-BLINKIT,Debit,270.00,680737.0,UPI,TXN968962
5,2024-01-01,14:48,BHIM ZEPTO,DR,Rs. 625,677650.0,UPI,TXN370902
6,2024-01-01,14:50,UPI-UBER-2426@HDFCBANK,Debit,Rs. 148,677020.0,UPI,TXN546173
7,01-Jan-24,21:44,BHIM-BLINKIT,Debit,₹482,677168.0,UPI,TXN599219
8,2024-01-02,05:18,POS SWIGGY BANGALORE,Debit,Rs. 537,676177.0,UPI,TXN293319
9,02/01/24,06:55,UPI-GROWWPAY@HDFCBANK,DR,3956,672221.0,UPI,TXN418594


In [67]:
# Fix Date column (4 mixed formats)
df['date'] = pd.to_datetime(df['Date'],format='mixed',dayfirst=True,errors='coerce')

# Fix Amount column (3 mixed formats)
df['amount'] = (
    df['Amount']
    .astype(str)
    .str.replace('₹', '', regex=False)
    .str.replace('Rs.', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)
df['amount'] = pd.to_numeric(df['amount'], errors='coerce')

# Standardise Type column
type_map = {
    'DR': 'debit',
    'CR': 'credit',
    'Debit': 'debit',
    'Credit': 'credit'
}
df['type_clean'] = df['Type'].map(type_map)

# Extract hour from Time
df['hour'] = df['Time'].str[:2].astype(int)

print("NaN dates right after cleaning:", df['date'].isna().sum())
print("NaN amounts right after cleaning:", df['amount'].isna().sum())
print(df['type_clean'].value_counts(dropna=False))

NaN dates right after cleaning: 0
NaN amounts right after cleaning: 0
type_clean
debit     1322
credit       6
Name: count, dtype: int64


In [68]:
before = df.shape[0]
df = df.drop_duplicates()
after_dedup = df.shape[0]

print("NaN dates before dropna:", df['date'].isna().sum())
print("NaN amounts before dropna:", df['amount'].isna().sum())

df = df.dropna(subset=['date', 'amount'])
after_dropna = df.shape[0]

df['Mode'] = df['Mode'].replace('', np.nan)

print("before:", before)
print("after_dedup:", after_dedup)
print("after_dropna:", after_dropna)

NaN dates before dropna: 0
NaN amounts before dropna: 0
before: 1328
after_dedup: 1310
after_dropna: 1310


In [69]:
print(f"Parsed {df.shape[0]} transactions across 6 months.")
print(f"Dropped {before - after_dedup} duplicates.")
print(f"Unparseable amounts: {df['amount'].isna().sum()}")
print(f"Unparseable dates: {df['date'].isna().sum()}")
print(df.dtypes)

Parsed 1310 transactions across 6 months.
Dropped 18 duplicates.
Unparseable amounts: 0
Unparseable dates: 0
Date                   object
Time                   object
Description            object
Type                   object
Amount                 object
Balance               float64
Mode                   object
Ref                    object
date           datetime64[ns]
amount                float64
type_clean             object
hour                    int64
dtype: object




---



# #Feature 2 - Vendor Extractor

In [70]:
pd.set_option('display.max_rows', None)
print(df['Description'].nunique())
print(df['Description'].unique())

283
['AMAZON SELLER SVCS' 'BHIM-BMTC' 'NEFT-TECHCRUSH LABS-SALARY MAY24'
 'UPI-AMAN-8934@OKAXIS' 'BHIM-BLINKIT' 'BHIM ZEPTO'
 'UPI-UBER-2426@HDFCBANK' 'POS SWIGGY BANGALORE' 'UPI-GROWWPAY@HDFCBANK'
 'OLA ELECTRIC' 'BMS MOVIE TICKETS' 'POS OLA-PRIME' 'SWIGGY-INSTAMART'
 'UPI-STARBUCKS@AXIS' 'UPI-THIRDWAVE@OKAXIS' 'ANI Technologies'
 'BMTC BUS PASS' 'POS TRUFFLES' 'FLIPKART INDIA' 'POS SWIGGY-RESTAURANT'
 'GROFERS INDIA P L' 'POS UBER BANGALORE' 'BANGALORE ELEC SUPPLY'
 'TWC INDIA' 'UPI-BESCOM-BILL@HDFCBANK' 'UPI-AMAN-0816@OKAXIS'
 'ROPPEN TRANSPORTATION' 'OLA CABS' 'POS ZOMATO' 'UPI-AMAZONPAY@HDFCBANK'
 'POS BLINKIT' 'IMPS-RENT-LANDLORD-75500265' 'ZOMATO MEDIA P L'
 'UPI-ANKIT-6430@OKAXIS' 'UPI-OLACABS@HDFCBANK' 'UPI-JIORECHARGE@PAYTM'
 'UPI-CCD@HDFCBANK' 'Swiggy*Order' 'INSTAMART BANGALORE'
 'UPI-ZOMATO-LIMITED@PAYTM' 'AVENUE SUPERMARTS' 'POS HP PETROL STATION'
 'UPI-VIKAS-6060@OKAXIS' 'POS BANGALORE RESTAURANT'
 'UPI-ZERODHA-COIN@AXIS' 'BHIM SWIGGY' 'UPI-BOOKMYSHOW@HDFCBANK'
 'BLINKIT

In [71]:
vendor_map = {
    # Quick Commerce (check INSTAMART before SWIGGY!)
    'Swiggy Instamart': ['INSTAMART'],
    'Zepto': ['ZEPTO', 'KIRANAKART'],
    'Blinkit': ['BLINKIT', 'GROFERS'],
    'BigBasket': ['BIGBASKET', 'INNOVATIVE RETAIL'],

    # Subscriptions (check before generic Amazon)
    'Netflix': ['NETFLIX'],
    'Spotify': ['SPOTIFY'],
    'Hotstar': ['HOTSTAR', 'STAR INDIA'],
    'Amazon Prime': ['AMZN PRIME', 'AMAZON PRIME'],

    # Ecommerce
    'Amazon': ['AMAZON', 'AMZN', 'FSN E-COMMERCE'],
    'Flipkart': ['FLIPKART', 'FKART'],
    'Myntra': ['MYNTRA'],
    'Nykaa': ['NYKAA'],

    # Food Delivery
    'Swiggy': ['SWIGGY', 'BUNDL'],
    'Zomato': ['ZOMATO'],
    'Zomato Dineout': ['DINEOUT'],

    # Transport
    'Uber': ['UBER'],
    'Ola': ['OLA', 'ANI TECHNOLOGIES'],
    'Rapido': ['RAPIDO', 'ROPPEN'],
    'BMTC': ['BMTC', 'TUMMOC'],

    # Cafe
    'Starbucks': ['STARBUCKS', 'TATA STARBUCKS'],
    'Third Wave Coffee': ['THIRDWAVE', 'TWC INDIA', 'THIRD WAVE'],
    'Cafe Coffee Day': ['CCD', 'COFFEE DAY'],

    # Restaurants
    'Restaurant': ['RESTAURANT', 'TRUFFLES', 'EMPIRE', 'MEGHANA'],

    # Investments
    'Zerodha': ['ZERODHA'],
    'Groww': ['GROWW', 'NEXTBILLION'],

    # Entertainment
    'BookMyShow': ['BOOKMYSHOW', 'BMS MOVIE', 'BIGTREE'],

    # Utilities
    'Airtel': ['AIRTEL'],
    'Vi': ['VI POSTPAID', 'VODAFONE', 'VI-RECHARGE'],
    'Jio': ['JIO'],
    'BESCOM': ['BESCOM', 'BANGALORE ELEC'],
    'BWSSB': ['BWSSB'],

    # Fuel
    'HP Petrol': ['HP PETROL'],
    'BPCL': ['BPCL'],
    'Indian Oil': ['INDIAN OIL', 'IOC'],

    # Groceries
    'DMart': ['DMART', 'AVENUE SUPERMARTS'],

    # Rent & Salary (non-category, but present in your data)
    'Rent': ['RENT-LANDLORD'],
    'Salary': ['SALARY'],
}

In [72]:
def extract_vendor(description):
    desc_upper = str(description).upper()

    # ATM withdrawals
    if 'ATM' in desc_upper:
        return 'Cash Withdrawal'

    # Check vendor dictionary (order matters — specific before generic)
    for canonical_name, keywords in vendor_map.items():
        if any(keyword in desc_upper for keyword in keywords):
            return canonical_name

    # P2P transfers — pattern: UPI-<NAME>-####@... with no vendor match
    if desc_upper.startswith('UPI-') and '@' in desc_upper:
        return 'P2P Transfer'

    return 'Uncategorised'

df['vendor_clean'] = df['Description'].apply(extract_vendor)

In [73]:
print("Unique vendors:", df['vendor_clean'].nunique())
print(df['vendor_clean'].value_counts())

Unique vendors: 39
vendor_clean
Swiggy               176
Zomato               121
Ola                   87
Amazon                80
Zepto                 71
Uber                  71
Swiggy Instamart      67
Restaurant            63
Rapido                55
Blinkit               55
Flipkart              47
Starbucks             42
BMTC                  37
Third Wave Coffee     31
Cafe Coffee Day       26
DMart                 22
Myntra                20
BigBasket             19
P2P Transfer          18
Cash Withdrawal       17
Indian Oil            17
Nykaa                 16
BESCOM                15
Zerodha               14
Hotstar               13
BookMyShow            13
Zomato Dineout        10
Netflix               10
HP Petrol              9
Groww                  9
Amazon Prime           9
Spotify                8
Vi                     8
BWSSB                  8
Salary                 6
Rent                   6
Jio                    6
Airtel                 6
BPCL              

In [74]:
uncategorised = df[df['vendor_clean'] == 'Uncategorised']['Description'].unique()
print("Uncategorised count:", len(uncategorised))
print(uncategorised)

Uncategorised count: 0
[]




---



# #Feature 3 - Category Tagger + Spending Overview

In [75]:
category_map = {
    # Food Delivery
    'Swiggy': 'Food Delivery',
    'Zomato': 'Food Delivery',
    'Zomato Dineout': 'Food Delivery',

    # Quick Commerce
    'Swiggy Instamart': 'Quick Commerce',
    'Zepto': 'Quick Commerce',
    'Blinkit': 'Quick Commerce',
    'BigBasket': 'Quick Commerce',

    # Ecommerce
    'Amazon': 'Ecommerce',
    'Amazon Prime': 'Subscriptions',   # Prime membership is a subscription, not a purchase
    'Flipkart': 'Ecommerce',
    'Myntra': 'Ecommerce',
    'Nykaa': 'Ecommerce',

    # Transport
    'Uber': 'Transport',
    'Ola': 'Transport',
    'Rapido': 'Transport',
    'BMTC': 'Transport',

    # Cafe
    'Starbucks': 'Cafe',
    'Third Wave Coffee': 'Cafe',
    'Cafe Coffee Day': 'Cafe',

    # Restaurants
    'Restaurant': 'Restaurants',

    # Groceries
    'DMart': 'Groceries',

    # Investments
    'Zerodha': 'Investments',
    'Groww': 'Investments',

    # Entertainment
    'BookMyShow': 'Entertainment',

    # Subscriptions
    'Hotstar': 'Subscriptions',
    'Netflix': 'Subscriptions',
    'Spotify': 'Subscriptions',

    # Utilities
    'BESCOM': 'Utilities',
    'BWSSB': 'Utilities',
    'Vi': 'Utilities',
    'Jio': 'Utilities',
    'Airtel': 'Utilities',

    # Fuel
    'Indian Oil': 'Fuel',
    'HP Petrol': 'Fuel',
    'BPCL': 'Fuel',

    # Special categories (not vendor-based spend)
    'P2P Transfer': 'Personal Transfer',
    'Cash Withdrawal': 'Cash Withdrawal',
    'Rent': 'Rent',
    'Salary': 'Salary',
}

df['category'] = df['vendor_clean'].map(category_map)

# sanity check — should be 0
print("Unmapped vendors:", df['category'].isna().sum())

Unmapped vendors: 0


In [76]:
print(df['category'].value_counts())

category
Food Delivery        307
Transport            250
Quick Commerce       212
Ecommerce            163
Cafe                  99
Restaurants           63
Utilities             43
Subscriptions         40
Fuel                  28
Investments           23
Groceries             22
Personal Transfer     18
Cash Withdrawal       17
Entertainment         13
Salary                 6
Rent                   6
Name: count, dtype: int64


In [77]:
total_credits = df[df['type_clean'] == 'credit']['amount'].sum()
total_debits = df[df['type_clean'] == 'debit']['amount'].sum()
net_change = total_credits - total_debits
savings_rate = (total_credits - total_debits) / total_credits * 100

print("="*60)
print(" EXECUTIVE SUMMARY")
print("="*60)
print(f" Total credits : Rs. {total_credits:,.0f}")
print(f" Total debits  : Rs. {total_debits:,.0f}")
print(f" Net change    : Rs. {net_change:,.0f}")
print(f" Savings rate  : {savings_rate:.1f}%")
print(f" Transactions  : {df.shape[0]}")
print(f" Unique vendors: {df['vendor_clean'].nunique()}")

 EXECUTIVE SUMMARY
 Total credits : Rs. 509,774
 Total debits  : Rs. 1,678,901
 Net change    : Rs. -1,169,127
 Savings rate  : -229.3%
 Transactions  : 1310
 Unique vendors: 39


In [78]:
# Exclude Personal Transfer and Cash Withdrawal from "spend" percentages (per brief's recommendation)
debit_df = df[(df['type_clean'] == 'debit') &
              (~df['category'].isin(['Personal Transfer', 'Cash Withdrawal', 'Rent']))]

top_categories = debit_df.groupby('category')['amount'].sum().sort_values(ascending=False)
total_debit_for_pct = top_categories.sum()

print(" TOP CATEGORIES (% of debit total)")
for cat, amt in top_categories.head(5).items():
    pct = amt / total_debit_for_pct * 100
    print(f" {cat:<20}{pct:>5.1f}%   Rs. {amt:,.0f}")

print("\n TOP VENDORS")
top_vendors = debit_df.groupby('vendor_clean')['amount'].agg(['sum', 'count']).sort_values('sum', ascending=False)
for vendor, row in top_vendors.head(5).iterrows():
    print(f" {vendor:<20}Rs. {row['sum']:>10,.0f} ({int(row['count'])} orders)")

 TOP CATEGORIES (% of debit total)
 Ecommerce            39.6%   Rs. 593,959
 Investments          16.5%   Rs. 248,160
 Food Delivery         9.9%   Rs. 148,879
 Quick Commerce        7.8%   Rs. 117,611
 Restaurants           6.5%   Rs. 97,912

 TOP VENDORS
 Amazon              Rs.    322,908 (80 orders)
 Zerodha             Rs.    210,000 (14 orders)
 Flipkart            Rs.    177,510 (47 orders)
 Restaurant          Rs.     97,912 (63 orders)
 Swiggy              Rs.     73,738 (176 orders)




---



# #Feature 4 - Spending Overview


In [79]:
# Executive Summary

total_credits = df[df['type_clean'] == 'credit']['amount'].sum()
total_debits = df[df['type_clean'] == 'debit']['amount'].sum()
net_change = total_credits - total_debits
savings_rate = (total_credits - total_debits) / total_credits * 100

print("="*60)
print(" EXECUTIVE SUMMARY")
print("="*60)
print(f" Total credits : Rs. {total_credits:,.0f}")
print(f" Total debits  : Rs. {total_debits:,.0f}")
print(f" Net change    : Rs. {net_change:,.0f}")
print(f" Savings rate  : {savings_rate:.1f}%")
print(f" Transactions  : {df.shape[0]}")
print(f" Unique vendors: {df['vendor_clean'].nunique()}")

 EXECUTIVE SUMMARY
 Total credits : Rs. 509,774
 Total debits  : Rs. 1,678,901
 Net change    : Rs. -1,169,127
 Savings rate  : -229.3%
 Transactions  : 1310
 Unique vendors: 39


In [80]:
# Top Categories and Top Vendors by Spend
debit_df = df[(df['type_clean'] == 'debit') &
              (~df['category'].isin(['Personal Transfer', 'Cash Withdrawal', 'Rent']))]

top_categories = debit_df.groupby('category')['amount'].sum().sort_values(ascending=False)
total_debit_for_pct = top_categories.sum()

print(" TOP CATEGORIES (% of debit total)")
for cat, amt in top_categories.head(5).items():
    pct = amt / total_debit_for_pct * 100
    print(f" {cat:<20}{pct:>5.1f}%   Rs. {amt:,.0f}")

print("\n TOP VENDORS")
top_vendors = debit_df.groupby('vendor_clean')['amount'].agg(['sum', 'count']).sort_values('sum', ascending=False)
for vendor, row in top_vendors.head(5).iterrows():
    print(f" {vendor:<20}Rs. {row['sum']:>10,.0f} ({int(row['count'])} orders)")

 TOP CATEGORIES (% of debit total)
 Ecommerce            39.6%   Rs. 593,959
 Investments          16.5%   Rs. 248,160
 Food Delivery         9.9%   Rs. 148,879
 Quick Commerce        7.8%   Rs. 117,611
 Restaurants           6.5%   Rs. 97,912

 TOP VENDORS
 Amazon              Rs.    322,908 (80 orders)
 Zerodha             Rs.    210,000 (14 orders)
 Flipkart            Rs.    177,510 (47 orders)
 Restaurant          Rs.     97,912 (63 orders)
 Swiggy              Rs.     73,738 (176 orders)




---



# #Feature 5 - Monthly Trend Analysis

In [81]:
df['month'] = df['date'].dt.month
debit_df = df[(df['type_clean'] == 'debit') &
              (~df['category'].isin(['Personal Transfer', 'Cash Withdrawal', 'Rent']))]

month_pivot = debit_df.pivot_table(
    values='amount',
    index='category',
    columns='month',
    aggfunc='sum',
    fill_value=0
)

print(month_pivot)

month                 1        2         3        4        5         6
category                                                              
Cafe             3690.0   4273.0    5448.0   6564.0   5668.0    5802.0
Ecommerce       97134.0  92773.0  103772.0  68098.0  95191.0  136991.0
Entertainment    1263.0    474.0    2418.0   2224.0      0.0    1914.0
Food Delivery   21527.0  25775.0   24032.0  25024.0  23945.0   28576.0
Fuel            30322.0   2079.0   26164.0  18718.0   9138.0    2882.0
Groceries       14594.0   3873.0    3658.0   3334.0   7227.0    4777.0
Investments     38432.0  15000.0   68644.0  54126.0  48628.0   23330.0
Quick Commerce  15852.0  22163.0   20610.0  24986.0  17679.0   16321.0
Restaurants     15683.0  17449.0   25131.0   5741.0  20508.0   13400.0
Subscriptions    4256.0   5866.0    7952.0   3289.0   2429.0    4597.0
Transport       11005.0  10191.0    6857.0   9284.0  13141.0    6996.0
Utilities        9157.0   6870.0    5497.0   5831.0   4340.0   10219.0


In [82]:
first_month = month_pivot.columns.min()
last_month = month_pivot.columns.max()

pct_change = ((month_pivot[last_month] - month_pivot[first_month]) / month_pivot[first_month].replace(0, np.nan)) * 100
pct_change = pct_change.sort_values(ascending=False)

print("Biggest growth:")
print(pct_change.head(1))
print("\nBiggest decline:")
print(pct_change.tail(1))

Biggest growth:
category
Cafe    57.235772
dtype: float64

Biggest decline:
category
Fuel   -90.49535
dtype: float64


In [83]:
month_names = {1:'Jan', 2:'Feb', 3:'Mar', 4:'Apr', 5:'May', 6:'Jun'}

print(" MONTHLY TREND (Food Delivery)")
food_trend = month_pivot.loc['Food Delivery']
for m, amt in food_trend.items():
    bar = "#" * int(amt / 3000)
    print(f" {month_names[m]}  Rs. {amt:>8,.0f}  {bar}")

 MONTHLY TREND (Food Delivery)
 Jan  Rs.   21,527  #######
 Feb  Rs.   25,775  ########
 Mar  Rs.   24,032  ########
 Apr  Rs.   25,024  ########
 May  Rs.   23,945  #######
 Jun  Rs.   28,576  #########




---



# #Feature 6 - Time-of-Day Patterns

In [84]:
hour_pivot = debit_df.pivot_table(
    values='amount',
    index='category',
    columns='hour',
    aggfunc='count',
    fill_value=0
)

print(hour_pivot)

hour            0   1   2   3   4   5   6   7   8   9   ...  14  15  16  17  \
category                                                ...                   
Cafe             3   3   0   1   0   0   1   2   9   7  ...   5   7   8  10   
Ecommerce        9   5   4   8   7   7   6   4   4   9  ...   9   5   8   7   
Entertainment    1   1   0   0   0   0   0   1   2   0  ...   0   0   2   0   
Food Delivery    1   7   2   5   6   7   1   2   8  10  ...   9  15  10  11   
Fuel             2   1   0   1   1   1   0   0   1   2  ...   1   2   3   0   
Groceries        2   1   1   3   1   0   1   0   1   1  ...   0   0   0   0   
Investments      1   1   0   0   2   1   4   0   0   1  ...   0   0   0   0   
Quick Commerce   4   3   3   5   2   3   4   2   5  17  ...  10   9  12   4   
Restaurants      1   1   1   0   0   2   1   0   2   1  ...   1   1   4   2   
Subscriptions    0   1   2   7  10   5   4   2   0   1  ...   1   0   0   0   
Transport        3   1   2   2   3   8   5  15  19  

In [85]:
food_orders = df[(df['category'] == 'Food Delivery')]
total_food_orders = food_orders.shape[0]

late_night = food_orders[(food_orders['hour'] >= 21) | (food_orders['hour'] <= 1)]
late_night_pct = late_night.shape[0] / total_food_orders * 100

print(f"Total Food Delivery orders: {total_food_orders}")
print(f"Late-night (9PM-1AM) orders: {late_night.shape[0]}")
print(f"Late-night percentage: {late_night_pct:.1f}%")

Total Food Delivery orders: 307
Late-night (9PM-1AM) orders: 62
Late-night percentage: 20.2%


In [86]:
cafe_orders = df[df['category'] == 'Cafe']
morning_cafe = cafe_orders[(cafe_orders['hour'] >= 8) & (cafe_orders['hour'] <= 11)]
morning_pct = morning_cafe.shape[0] / cafe_orders.shape[0] * 100

print(f"Total Cafe orders: {cafe_orders.shape[0]}")
print(f"Morning (8-11AM) Cafe orders: {morning_cafe.shape[0]}")
print(f"Morning percentage: {morning_pct:.1f}%")

Total Cafe orders: 99
Morning (8-11AM) Cafe orders: 35
Morning percentage: 35.4%


In [87]:
print(" TIME-OF-DAY PATTERNS")
print(f" Food Delivery: {late_night_pct:.1f}% late-night (9PM-1AM), {evening_window.shape[0] / total_food_orders * 100:.1f}% evening (6PM-11PM) - dinner-hour pattern, not late-night")
print(f" Cafe: {morning_pct:.1f}% morning (8-11AM) - moderate morning coffee tendency")

 TIME-OF-DAY PATTERNS
 Food Delivery: 20.2% late-night (9PM-1AM), 50.2% evening (6PM-11PM) - dinner-hour pattern, not late-night
 Cafe: 35.4% morning (8-11AM) - moderate morning coffee tendency




---



# #Feature 7 - Anomaly Detection


In [88]:
category_mean = df.groupby('category')['amount'].transform('mean')
category_std = df.groupby('category')['amount'].transform('std')
df['z_score'] = (df['amount'] - category_mean) / category_std

anomalies = df[df['z_score'] > 2].sort_values('z_score', ascending=False)
print(f"Total anomalies flagged: {anomalies.shape[0]}")

Total anomalies flagged: 36


In [89]:
print(" TOP ANOMALIES (2+ stddev from category mean)")
for _, row in anomalies.head(10).iterrows():
    date_str = row['date'].strftime('%d %b')
    print(f" {date_str} - {row['vendor_clean']:<15} Rs. {row['amount']:>8,.0f}  (z={row['z_score']:.1f})  [{row['category']}]")

 TOP ANOMALIES (2+ stddev from category mean)
 22 Jun - Zomato Dineout  Rs.    7,935  (z=15.5)  [Food Delivery]
 12 Mar - BigBasket       Rs.    1,865  (z=4.3)  [Quick Commerce]
 20 May - BigBasket       Rs.    1,856  (z=4.3)  [Quick Commerce]
 23 Jan - BigBasket       Rs.    1,797  (z=4.1)  [Quick Commerce]
 26 Feb - Restaurant      Rs.    8,383  (z=4.0)  [Restaurants]
 26 Jun - Amazon          Rs.   22,008  (z=4.0)  [Ecommerce]
 07 Feb - Amazon          Rs.   21,986  (z=4.0)  [Ecommerce]
 31 Mar - Restaurant      Rs.    7,931  (z=3.8)  [Restaurants]
 08 Apr - BigBasket       Rs.    1,682  (z=3.7)  [Quick Commerce]
 05 Mar - Amazon          Rs.   19,917  (z=3.5)  [Ecommerce]




---



# #Feature 8 - Spending Archetype Detection

In [90]:
# Category % of total debit spend (excluding transfers/withdrawals/rent)
category_pct = (top_categories / total_debit_for_pct * 100)

foodie_pct = category_pct.get('Food Delivery', 0) + category_pct.get('Restaurants', 0) + category_pct.get('Cafe', 0)
qcom_pct = category_pct.get('Quick Commerce', 0)
shop_pct = category_pct.get('Ecommerce', 0)
invest_pct = category_pct.get('Investments', 0)
transport_pct = category_pct.get('Transport', 0)

# Late-night snacker uses your actual evening finding
late_night_snacker_pct = late_night_pct  # from Feature 6

# Subscription vendor count
subscription_vendors = df[df['category'] == 'Subscriptions']['vendor_clean'].nunique()

print(f"Foodie %: {foodie_pct:.1f}")
print(f"Quick Commerce %: {qcom_pct:.1f}")
print(f"Shopaholic %: {shop_pct:.1f}")
print(f"Investor %: {invest_pct:.1f}")
print(f"Transport %: {transport_pct:.1f}")
print(f"Late-night snacker %: {late_night_snacker_pct:.1f}")
print(f"Subscription vendors: {subscription_vendors}")
print(f"Savings rate: {savings_rate:.1f}")

Foodie %: 18.5
Quick Commerce %: 7.8
Shopaholic %: 39.6
Investor %: 16.5
Transport %: 3.8
Late-night snacker %: 20.2
Subscription vendors: 4
Savings rate: -229.3


In [91]:
def is_foodie(pct):
    return pct > 25

def is_quick_commerce_junkie(pct):
    return pct > 15

def is_shopaholic(pct):
    return pct > 15

def is_investor(pct):
    return pct > 15

def is_late_night_snacker(pct):
    return pct > 50

def is_cab_commuter(pct):
    return pct > 10

def is_subscription_lover(count):
    return count >= 5

def is_yolo_spender(rate):
    return rate < 10

def is_disciplined_saver(rate):
    return rate > 40

# Bonus invented archetype — see Cell 5 below
def is_dinner_hour_orderer(evening_pct):
    return evening_pct > 45

In [92]:
print(" RAHUL'S SPENDING ARCHETYPES")

if is_foodie(foodie_pct):
    print(f" -> THE FOODIE ({foodie_pct:.1f}% on food)")

if is_quick_commerce_junkie(qcom_pct):
    print(f" -> THE QUICK COMMERCE JUNKIE ({qcom_pct:.1f}% on Q-com)")

if is_shopaholic(shop_pct):
    print(f" -> THE SHOPAHOLIC ({shop_pct:.1f}% on e-commerce)")

if is_investor(invest_pct):
    print(f" -> THE INVESTOR ({invest_pct:.1f}% on SIPs)")

if is_late_night_snacker(late_night_snacker_pct):
    print(f" -> THE LATE-NIGHT SNACKER ({late_night_snacker_pct:.1f}% food after 9PM)")

if is_cab_commuter(transport_pct):
    print(f" -> THE CAB COMMUTER ({transport_pct:.1f}% on transport)")

if is_subscription_lover(subscription_vendors):
    print(f" -> THE SUBSCRIPTION LOVER ({subscription_vendors} active subs)")

if is_yolo_spender(savings_rate):
    print(f" -> THE YOLO SPENDER (savings rate {savings_rate:.1f}%)")

if is_disciplined_saver(savings_rate):
    print(f" -> THE DISCIPLINED SAVER (savings rate {savings_rate:.1f}%)")

 RAHUL'S SPENDING ARCHETYPES
 -> THE SHOPAHOLIC (39.6% on e-commerce)
 -> THE INVESTOR (16.5% on SIPs)
 -> THE YOLO SPENDER (savings rate -229.3%)


In [93]:
evening_pct = evening_window.shape[0] / total_food_orders * 100

print("\n BONUS ARCHETYPE")
if is_dinner_hour_orderer(evening_pct):
    print(f" -> THE BENGALURU DINNER-HOUR ORDERER ({evening_pct:.1f}% of food orders between 6-11 PM)")
    print("    Detection rule: >45% of Food Delivery orders fall between 18:00-23:00.")
    print("    Unlike the stereotypical late-night snacker, this profile reflects Bangalore's")
    print("    tech commute culture — long office hours mean dinner gets ordered in, not cooked,")
    print("    right after getting home from work rather than as a late-night impulse.")


 BONUS ARCHETYPE
 -> THE BENGALURU DINNER-HOUR ORDERER (50.2% of food orders between 6-11 PM)
    Detection rule: >45% of Food Delivery orders fall between 18:00-23:00.
    Unlike the stereotypical late-night snacker, this profile reflects Bangalore's
    tech commute culture — long office hours mean dinner gets ordered in, not cooked,
    right after getting home from work rather than as a late-night impulse.




---



# #Final Report

In [94]:
print("="*64)
print(" SpendDNA REPORT - RAHUL SHARMA")
print(" 6 months - {} transactions - Jan to Jun 2024".format(df.shape[0]))
print("="*64)

print("\n EXECUTIVE SUMMARY")
print(f" Total credits : Rs. {total_credits:,.0f}")
print(f" Total debits  : Rs. {total_debits:,.0f}")
print(f" Net change    : Rs. {net_change:,.0f}")
print(f" Savings rate  : {savings_rate:.1f}%")
print(f" Transactions  : {df.shape[0]}")
print(f" Unique vendors: {df['vendor_clean'].nunique()}")

print("\n TOP CATEGORIES (% of debit total)")
for cat, amt in top_categories.head(5).items():
    pct = amt / total_debit_for_pct * 100
    bar = "#" * int(pct / 2)
    print(f" {cat:<18}{pct:>5.1f}%  Rs. {amt:>9,.0f}  {bar}")

print("\n TOP VENDORS")
for vendor, row in top_vendors.head(5).iterrows():
    print(f" {vendor:<18}Rs. {row['sum']:>9,.0f}  ({int(row['count'])} orders)")

print("\n TIME-OF-DAY PATTERNS")
print(f" Food Delivery: {late_night_pct:.1f}% late-night (9PM-1AM), {evening_pct:.1f}% evening (6PM-11PM)")
print(f" Cafe: {morning_pct:.1f}% morning (8-11AM)")

print("\n MONTHLY TREND (Food Delivery)")
for m, amt in food_trend.items():
    bar = "#" * int(amt / 3000)
    print(f" {month_names[m]}  Rs. {amt:>8,.0f}  {bar}")

print("\n TOP ANOMALIES (2+ stddev from category mean)")
for _, row in anomalies.head(5).iterrows():
    date_str = row['date'].strftime('%d %b')
    print(f" {date_str} - {row['vendor_clean']:<15} Rs. {row['amount']:>8,.0f} (z={row['z_score']:.1f})")

print("\n RAHUL'S SPENDING ARCHETYPES")
if is_shopaholic(shop_pct):
    print(f" -> THE SHOPAHOLIC ({shop_pct:.1f}% on e-commerce)")
if is_investor(invest_pct):
    print(f" -> THE INVESTOR ({invest_pct:.1f}% on SIPs)")
if is_yolo_spender(savings_rate):
    print(f" -> THE YOLO SPENDER (savings rate {savings_rate:.1f}%)")
if is_dinner_hour_orderer(evening_pct):
    print(f" -> THE BENGALURU DINNER-HOUR ORDERER ({evening_pct:.1f}% evening food orders)")

print("="*64)
print(" KEY INSIGHTS")
print(f" 1. Rahul is burning through savings at a rate that puts him")
print(f"    {abs(savings_rate):.0f}% over his income - unsustainable long-term.")
print(f" 2. E-commerce ({shop_pct:.1f}%) is his single biggest discretionary")
print(f"    spend, driven by a few large anomalous orders (up to Rs. 22,008).")
print(f" 3. Unlike a typical late-night spender, {evening_pct:.1f}% of his food")
print(f"    orders land in the 6-11 PM window - a dinner-hour habit tied")
print(f"    to Bengaluru's long tech-commute culture, not late-night stress.")
print("="*64)

 SpendDNA REPORT - RAHUL SHARMA
 6 months - 1310 transactions - Jan to Jun 2024

 EXECUTIVE SUMMARY
 Total credits : Rs. 509,774
 Total debits  : Rs. 1,678,901
 Net change    : Rs. -1,169,127
 Savings rate  : -229.3%
 Transactions  : 1310
 Unique vendors: 39

 TOP CATEGORIES (% of debit total)
 Ecommerce          39.6%  Rs.   593,959  ###################
 Investments        16.5%  Rs.   248,160  ########
 Food Delivery       9.9%  Rs.   148,879  ####
 Quick Commerce      7.8%  Rs.   117,611  ###
 Restaurants         6.5%  Rs.    97,912  ###

 TOP VENDORS
 Amazon            Rs.   322,908  (80 orders)
 Zerodha           Rs.   210,000  (14 orders)
 Flipkart          Rs.   177,510  (47 orders)
 Restaurant        Rs.    97,912  (63 orders)
 Swiggy            Rs.    73,738  (176 orders)

 TIME-OF-DAY PATTERNS
 Food Delivery: 20.2% late-night (9PM-1AM), 50.2% evening (6PM-11PM)
 Cafe: 35.4% morning (8-11AM)

 MONTHLY TREND (Food Delivery)
 Jan  Rs.   21,527  #######
 Feb  Rs.   25,775  ######



---



# #Bonus : Vendor Cleanup Audit

In [95]:
uncategorised = df[df['vendor_clean'] == 'Uncategorised']['Description'].unique()
print(f"Descriptions that failed vendor mapping: {len(uncategorised)}")
print(uncategorised)

Descriptions that failed vendor mapping: 0
[]


# #Bonus : Day-of-Week Analysis

In [96]:
df['day_of_week'] = df['date'].dt.day_name()

# Recreate debit_df here so it includes the new 'day_of_week' column
debit_df = df[(df['type_clean'] == 'debit') &
              (~df['category'].isin(['Personal Transfer', 'Cash Withdrawal', 'Rent']))]

weekday_spend = debit_df[debit_df['day_of_week'].isin(['Monday','Tuesday','Wednesday','Thursday','Friday'])]['amount'].sum()
weekend_spend = debit_df[debit_df['day_of_week'].isin(['Saturday','Sunday'])]['amount'].sum()

weekday_count = debit_df[debit_df['day_of_week'].isin(['Monday','Tuesday','Wednesday','Thursday','Friday'])].shape[0]
weekend_count = debit_df[debit_df['day_of_week'].isin(['Saturday','Sunday'])].shape[0]

weekday_avg = weekday_spend / weekday_count
weekend_avg = weekend_spend / weekend_count

pct_diff = (weekend_avg - weekday_avg) / weekday_avg * 100

print(f"Weekday avg transaction: Rs. {weekday_avg:,.0f}")
print(f"Weekend avg transaction: Rs. {weekend_avg:,.0f}")
print(f"Weekends are {pct_diff:.1f}% {'more' if pct_diff > 0 else 'less'} expensive per transaction than weekdays")

Weekday avg transaction: Rs. 1,212
Weekend avg transaction: Rs. 1,133
Weekends are -6.6% less expensive per transaction than weekdays




---



# #Bonus - Spend Forecasting (Next Month Projection)

In [97]:
# Recreate debit_df to make sure it's current
debit_df = df[(df['type_clean'] == 'debit') &
              (~df['category'].isin(['Personal Transfer', 'Cash Withdrawal', 'Rent']))]

month_pivot_forecast = debit_df.pivot_table(
    values='amount',
    index='category',
    columns='month',
    aggfunc='sum',
    fill_value=0
)

print("FORECASTED SPEND FOR NEXT MONTH (rolling 3-month average)")
print("="*55)

forecasts = {}
for category in month_pivot_forecast.index:
    monthly_values = month_pivot_forecast.loc[category].values  # NumPy array of 6 monthly totals
    last_3_months = monthly_values[-3:]  # April, May, June
    forecast = np.mean(last_3_months)
    forecasts[category] = forecast

# Sort by forecast amount, descending
forecasts_sorted = dict(sorted(forecasts.items(), key=lambda x: x[1], reverse=True))

for category, amt in forecasts_sorted.items():
    print(f" {category:<18} Rs. {amt:>10,.0f}")

total_forecast = sum(forecasts.values())
print("-"*55)
print(f" {'TOTAL PROJECTED':<18} Rs. {total_forecast:>10,.0f}")

FORECASTED SPEND FOR NEXT MONTH (rolling 3-month average)
 Ecommerce          Rs.    100,093
 Investments        Rs.     42,028
 Food Delivery      Rs.     25,848
 Quick Commerce     Rs.     19,662
 Restaurants        Rs.     13,216
 Fuel               Rs.     10,246
 Transport          Rs.      9,807
 Utilities          Rs.      6,797
 Cafe               Rs.      6,011
 Groceries          Rs.      5,113
 Subscriptions      Rs.      3,438
 Entertainment      Rs.      1,379
-------------------------------------------------------
 TOTAL PROJECTED    Rs.    243,639




---



## Reflection

This project pushed me to handle real-world messy data — four date formats, three amount formats,
and merchant names buried inside payment gateway strings. The hardest part was building the vendor
extraction dictionary without regex, which forced me to actually read through every unique description
rather than pattern-match blindly. I also learned that random synthetic datasets can vary meaningfully
from illustrative examples (my Ecommerce spend and dinner-hour food ordering pattern differed from the
brief's sample), which taught me to trust my verified logic over forcing my output to match a reference.

